In [ ]:

!pip install opencv-python


EXTRACTING DATA
::::::::::::::::::::::::::::::
::::::::::::::::::::::::::::::

In [ ]:
#extract from dataset
import os

your_dataset = []
base_folder = "GTSRB/Train"  # path to your Train folder

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))  # label is the folder name as integer




NORMAL LOGISTIC REGRESSION
::::::::::::::::::::::::::
:::::::::::::::::::::::::::


In [ ]:
#logistic regression model train
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score, accuracy_score, precision_score
import numpy as np
import cv2
import os
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
X = [] 
y = [] 

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64,64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(multi_class='multinomial', max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred = model.predict(X_test)
#report = classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)])
#print(report)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


FIRST TEST
:::::::::::::::::::;
::::::::::::::::::::

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
test_folder = "../RoadsignRecognition/GTSRB/Test"
for filename in os.listdir(test_folder):
    if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])
        print(f"Image: {filename}, Predicted Label: {predicted_label[0]},label_map[predicted_label]")
print()

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()
plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Print: Top Confused Pairs ---
#


ACTUAL TEST
:::::::::::::::::
:::::::::::::::::


In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]

# --- Predict on External Images ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")
print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


DATA AUGMENTATION AND LOGISTIC REGRESSION
::::::::::::::::::::::::::
::::::::::::::::::::::::::

In [ ]:
import numpy as np
import cv2

# Weather effect functions
def add_fog(image, intensity=0.9):
    h, w = image.shape
    fog = np.random.normal(loc=240, scale=25, size=(h, w)).astype(np.uint8)
    fog = cv2.GaussianBlur(fog, (151, 151), 0)  # large blur kernel
    foggy = cv2.addWeighted(image, 1 - intensity, fog, intensity, 0)
    return np.clip(foggy, 0, 255).astype(np.uint8)




def add_rain(image, drops=100):
    rain_img = image.copy()
    for _ in range(drops):
        x = np.random.randint(0, image.shape[1])
        y = np.random.randint(0, image.shape[0])
        length = np.random.randint(5, 15)
        cv2.line(rain_img, (x, y), (x, y + length), (255,), 1)
    return cv2.blur(rain_img, (3, 3))
def night_effect(image, factor=0.4):
    h, w = image.shape
    X_resultant_kernel = cv2.getGaussianKernel(w, 100)
    Y_resultant_kernel = cv2.getGaussianKernel(h, 100)
    kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * kernel / np.linalg.norm(kernel)
    night = image * factor * mask
    return np.clip(night, 0, 255).astype(np.uint8)



def add_glare(image, strength=0.6):
    overlay = image.copy()
    h, w = image.shape
    center = (np.random.randint(0, w), np.random.randint(0, h))
    radius = np.random.randint(30, 60)
    cv2.circle(overlay, center, radius, (255,), -1)
    return cv2.addWeighted(image, 1 - strength, overlay, strength, 0)

# Apply a weather effect batch-wise
def apply_effect_batch(X, effect_fn):
    transformed = []
    for x in X:
        img = (x * 255).reshape(64, 64).astype(np.uint8)
        effected = effect_fn(img)
        effected_flat = effected.flatten() / 255.0
        transformed.append(effected_flat)
    return np.array(transformed)

# Assuming you already have X_test and y_test from your dataset split
X_test_foggy = apply_effect_batch(X_test, add_fog)
X_test_rainy = apply_effect_batch(X_test, add_rain)
X_test_night = apply_effect_batch(X_test, night_effect)
X_test_glare = apply_effect_batch(X_test, add_glare)

# Predict and evaluate
from sklearn.metrics import accuracy_score

y_pred_normal = model.predict(X_test)
y_pred_foggy = model.predict(X_test_foggy)
y_pred_rainy = model.predict(X_test_rainy)
y_pred_night = model.predict(X_test_night)
y_pred_glare = model.predict(X_test_glare)

acc_normal = accuracy_score(y_test, y_pred_normal)
acc_foggy = accuracy_score(y_test, y_pred_foggy)
acc_rainy = accuracy_score(y_test, y_pred_rainy)
acc_night = accuracy_score(y_test, y_pred_night)
acc_glare = accuracy_score(y_test, y_pred_glare)

print("Normal Accuracy:", acc_normal)
print("Foggy Accuracy:", acc_foggy)
print("Rainy Accuracy:", acc_rainy)
print("Night Accuracy:", acc_night)
print("Glare Accuracy:", acc_glare)


AUGMENTATION DATA
::::::::::::::::::::
::::::::::::::::::::::

In [ ]:
import matplotlib.pyplot as plt

# Choose a random index from test set
idx = np.random.randint(0, len(X_test))
original_img = (X_test[idx] * 255).reshape(64, 64).astype(np.uint8)

# Apply weather effects
foggy_img = add_fog(original_img)
rainy_img = add_rain(original_img)
night_img = night_effect(original_img)
glare_img = add_glare(original_img)

# Plot all images
titles = ["Original", "Foggy", "Rainy", "Night", "Glare"]
images = [original_img, foggy_img, rainy_img, night_img, glare_img]

plt.figure(figsize=(15, 3))
for i, (title, img) in enumerate(zip(titles, images)):
    plt.subplot(1, 5, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()


FAILED LOGISTIC REGRESSION
::::::::::::::::::::::::::::
:::::::::::::::::::::::::::::

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# --- Weather Effect Functions (Same as before) ---
def add_fog(image, intensity=0.9):
    h, w = image.shape
    fog = np.random.normal(loc=240, scale=25, size=(h, w)).astype(np.uint8)
    fog = cv2.GaussianBlur(fog, (151, 151), 0)
    foggy = cv2.addWeighted(image, 1 - intensity, fog, intensity, 0)
    return np.clip(foggy, 0, 255).astype(np.uint8)

def add_rain(image, drops=100):
    rain_img = image.copy()
    for _ in range(drops):
        x = np.random.randint(0, image.shape[1])
        y = np.random.randint(0, image.shape[0])
        length = np.random.randint(5, 15)
        cv2.line(rain_img, (x, y), (x, y + length), (220,), 1)
    return cv2.blur(rain_img, (3, 3))

def night_effect(image, factor=0.4):
    h, w = image.shape
    X_resultant_kernel = cv2.getGaussianKernel(w, 100)
    Y_resultant_kernel = cv2.getGaussianKernel(h, 100)
    kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * kernel / np.linalg.norm(kernel)
    night = image.astype(float) * factor * (mask / 255.0)
    return np.clip(night, 0, 255).astype(np.uint8)

def add_glare(image, strength=0.6):
    overlay = image.copy()
    h, w = image.shape
    center = (np.random.randint(0, w), np.random.randint(0, h))
    radius = np.random.randint(30, 60)
    cv2.circle(overlay, center, radius, (255,), -1)
    overlay = cv2.GaussianBlur(overlay, (99,99), 0)
    return cv2.addWeighted(image, 1 - strength, overlay, strength, 0)

# --- Batch Application Function (Same as before) ---
def apply_effect_batch(X, effect_fn):
    transformed = []
    for x in X:
        img = (x * 255).reshape(64, 64).astype(np.uint8)
        effected = effect_fn(img)
        effected_flat = effected.flatten() / 255.0
        transformed.append(effected_flat)
    return np.array(transformed)


# --- Step 1: Load REAL GTSRB Data (Your Code) ---
print("Step 1: Loading GTSRB data from disk...")
your_dataset = []
base_folder = "GTSRB/Train"  # <--- MAKE SURE THIS PATH IS CORRECT

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm"): # GTSRB uses .ppm files
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))

X_data = []
y_data = []

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (64, 64))
        X_data.append(img.flatten() / 255.0)
        y_data.append(label)

X = np.array(X_data)
y = np.array(y_data)
print(f"Loaded {len(X)} images. Data shape: {X.shape}")
print("-" * 30)


# --- Step 2: Split Data into Training and Testing Sets ---
print("Step 2: Splitting data into training and test sets...")
# Using stratify=y is important for imbalanced datasets like GTSRB
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Data shapes: X_train: {X_train.shape}, X_test: {X_test.shape}")
print("-" * 30)


# --- Step 3: Create Augmented Datasets for Training and Testing ---
print("Step 3: Augmenting data with weather effects...")
# Augment the TRAINING data
X_train_foggy = apply_effect_batch(X_train, add_fog)
X_train_rainy = apply_effect_batch(X_train, add_rain)

# Augment the TEST data
X_test_foggy = apply_effect_batch(X_test, add_fog)
X_test_rainy = apply_effect_batch(X_test, add_rain)
X_test_night = apply_effect_batch(X_test, night_effect)
X_test_glare = apply_effect_batch(X_test, add_glare)
print("Data augmentation complete.")
print("-" * 30)


# --- Step 4: Combine Datasets for Robust Training ---
print("Step 4: Combining original and augmented training data...")
X_train_combined = np.vstack([X_train, X_train_foggy, X_train_rainy])
y_train_combined = np.concatenate([y_train, y_train, y_train])

# Shuffle the combined dataset
shuffle_idx = np.random.permutation(len(X_train_combined))
X_train_combined = X_train_combined[shuffle_idx]
y_train_combined = y_train_combined[shuffle_idx]

print(f"Combined training data shape: {X_train_combined.shape}")
print("-" * 30)


# --- Step 5: Train the ROBUST Logistic Regression Model ---
print("Step 5: Training the robust Logistic Regression model...")
# INCREASED max_iter to prevent ConvergenceWarning on the larger dataset
lr_model = LogisticRegression(
    multi_class='multinomial',
    max_iter=3000, # Increased from 1000
    random_state=42,
    solver='saga' # Good solver for large datasets
)
lr_model.fit(X_train_combined, y_train_combined)
print("Model training complete.")
print("-" * 30)


# --- Step 6: Evaluate the New Model on All Test Conditions ---
print("Step 6: Evaluating the new model...")
y_pred_normal = lr_model.predict(X_test)
y_pred_foggy = lr_model.predict(X_test_foggy)
y_pred_rainy = lr_model.predict(X_test_rainy)
y_pred_night = lr_model.predict(X_test_night)
y_pred_glare = lr_model.predict(X_test_glare)

accuracies = {
    "Normal": accuracy_score(y_test, y_pred_normal),
    "Foggy": accuracy_score(y_test, y_pred_foggy),
    "Rainy": accuracy_score(y_test, y_pred_rainy),
    "Night": accuracy_score(y_test, y_pred_night),
    "Glare": accuracy_score(y_test, y_pred_glare)
}

print("--- Robust Model Performance ---")
for condition, acc in accuracies.items():
    print(f"{condition} Accuracy: {acc:.4f}")
print("-" * 30)


# --- Step 7: Visualization ---
print("Step 7: Generating visualizations...")

# Accuracy Bar Chart
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))
conditions = list(accuracies.keys())
scores = list(accuracies.values())
bars = ax.bar(conditions, scores, color=['#4CAF50', '#FFC107', '#2196F3', '#3F51B5', '#FF5722'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Robust Logistic Regression Performance on GTSRB', fontsize=16, pad=20)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.3f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

# Confusion Matrix Heatmap (for the 'Normal' test case)
cm = confusion_matrix(y_test, y_pred_normal)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix on Normal (Unaffected) Test Data', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.show()

FAILED AUGMENTED LOGISTIC REGRESSION
::::::::::::::::::::::::::::::::::::
::::::::::::::::::::::::::::::::::::

In [ ]:
# --- Block 1: Imports ---
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# --- Block 2: Weather Effect Functions ---
def add_fog(image, intensity=0.9):
    """Adds a dense fog effect to a grayscale image."""
    h, w = image.shape
    fog = np.random.normal(loc=240, scale=25, size=(h, w)).astype(np.uint8)
    fog = cv2.GaussianBlur(fog, (151, 151), 0)
    foggy = cv2.addWeighted(image, 1 - intensity, fog, intensity, 0)
    return np.clip(foggy, 0, 255).astype(np.uint8)

def add_rain(image, drops=100):
    """Adds rain streaks to a grayscale image."""
    rain_img = image.copy()
    for _ in range(drops):
        x = np.random.randint(0, image.shape[1])
        y = np.random.randint(0, image.shape[0])
        length = np.random.randint(5, 15)
        cv2.line(rain_img, (x, y), (x, y + length), (220,), 1)
    return cv2.blur(rain_img, (3, 3))

def night_effect(image, factor=0.4):
    """Simulates a night-time view with a vignette effect."""
    h, w = image.shape
    X_resultant_kernel = cv2.getGaussianKernel(w, 100)
    Y_resultant_kernel = cv2.getGaussianKernel(h, 100)
    kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * kernel / np.linalg.norm(kernel)
    night = image.astype(float) * factor * (mask / 255.0)
    return np.clip(night, 0, 255).astype(np.uint8)

def add_glare(image, strength=0.6):
    """Adds a bright, blurry glare spot to a grayscale image."""
    overlay = image.copy()
    h, w = image.shape
    center = (np.random.randint(0, w), np.random.randint(0, h))
    radius = np.random.randint(30, 60)
    cv2.circle(overlay, center, radius, (255,), -1)
    overlay = cv2.GaussianBlur(overlay, (99,99), 0)
    return cv2.addWeighted(image, 1 - strength, overlay, strength, 0)

# --- Block 3: Batch Application Helper Function ---
def apply_effect_batch(X, effect_fn):
    """Applies a given effect function to a batch of flattened images."""
    transformed = []
    print(f"Applying effect: {effect_fn.__name__}...")
    for x in X:
        img = (x * 255).reshape(64, 64).astype(np.uint8)
        effected = effect_fn(img)
        effected_flat = effected.flatten() / 255.0
        transformed.append(effected_flat)
    return np.array(transformed)

# --- Block 4: Load REAL GTSRB Data with Error Checking ---
print("--- Step 1: Loading GTSRB data from disk ---")
your_dataset = []
base_folder = "GTSRB/Train"  # Ensure this path is correct

if not os.path.isdir(base_folder):
    print(f"ERROR: The folder '{base_folder}' was not found.")
    print("Please make sure the path is correct or the script is in the right directory.")
    sys.exit()

found_files = False
for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.lower().endswith(('.ppm', '.png', '.jpg', '.jpeg')):
                found_files = True
                img_path = os.path.join(label_path, filename)
                try:
                    your_dataset.append((img_path, int(label_folder)))
                except ValueError:
                    # Ignore folders with non-integer names (like .DS_Store)
                    continue

if not found_files:
    print(f"ERROR: No image files (.ppm, .png, etc.) found in the subdirectories of '{base_folder}'.")
    sys.exit()

X_data = []
y_data = []
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (64, 64))
        X_data.append(img.flatten() / 255.0)
        y_data.append(label)

X = np.array(X_data)
y = np.array(y_data)
print(f"Successfully loaded {len(X)} images. Data shape: {X.shape}")
print("-" * 30)

# --- Block 5: Split Data into Training and Testing Sets ---
print("--- Step 2: Splitting data into training and test sets ---")
# stratify=y is crucial for imbalanced datasets like GTSRB
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Data shapes: X_train: {X_train.shape}, X_test: {X_test.shape}")
print("-" * 30)

# --- Block 6: Create Augmented Datasets ---
print("--- Step 3: Augmenting data with weather effects (this may take a while) ---")
# Augment the TRAINING data to teach the model
X_train_foggy = apply_effect_batch(X_train, add_fog)
X_train_rainy = apply_effect_batch(X_train, add_rain)
# Augment the TEST data to evaluate performance
X_test_foggy = apply_effect_batch(X_test, add_fog)
X_test_rainy = apply_effect_batch(X_test, add_rain)
X_test_night = apply_effect_batch(X_test, night_effect)
X_test_glare = apply_effect_batch(X_test, add_glare)
print("Data augmentation complete.")
print("-" * 30)

# --- Block 7: Combine Datasets for Robust Training ---
print("--- Step 4: Combining original and augmented training data ---")
X_train_combined = np.vstack([X_train, X_train_foggy, X_train_rainy])
y_train_combined = np.concatenate([y_train, y_train, y_train])

# Shuffle the combined dataset to ensure random order during training
shuffle_idx = np.random.permutation(len(X_train_combined))
X_train_combined = X_train_combined[shuffle_idx]
y_train_combined = y_train_combined[shuffle_idx]
print(f"Combined training data shape: {X_train_combined.shape}")
print("-" * 30)

# --- Block 8: Train the ROBUST Logistic Regression Model ---
print("--- Step 5: Training the robust model (this is the longest step) ---")
# Increased max_iter and used 'saga' solver for the large, complex dataset
lr_model = LogisticRegression(
    multi_class='multinomial', max_iter=3000, random_state=42, solver='saga', n_jobs=-1
)
lr_model.fit(X_train_combined, y_train_combined)
print("Model training complete.")
print("-" * 30)

# --- Block 9: Evaluate the New Model on All Test Conditions ---
print("--- Step 6: Evaluating the new model ---")
y_pred_normal = lr_model.predict(X_test)
y_pred_foggy = lr_model.predict(X_test_foggy)
y_pred_rainy = lr_model.predict(X_test_rainy)
y_pred_night = lr_model.predict(X_test_night)
y_pred_glare = lr_model.predict(X_test_glare)

accuracies = {
    "Normal": accuracy_score(y_test, y_pred_normal),
    "Foggy": accuracy_score(y_test, y_pred_foggy),
    "Rainy": accuracy_score(y_test, y_pred_rainy),
    "Night": accuracy_score(y_test, y_pred_night),
    "Glare": accuracy_score(y_test, y_pred_glare)
}

print("\n--- Final Robust Model Performance ---")
for condition, acc in accuracies.items():
    print(f"{condition} Accuracy: {acc:.4f}")
print("-" * 30)

# --- Block 10: Visualization ---
print("--- Step 7: Generating visualizations ---")

# Accuracy Bar Chart
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))
conditions = list(accuracies.keys())
scores = list(accuracies.values())
bars = ax.bar(conditions, scores, color=['#4CAF50', '#FFC107', '#2196F3', '#3F51B5', '#FF5722'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Robust Logistic Regression Performance on GTSRB', fontsize=16, pad=20)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.3f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_normal)
plt.figure(figsize=(12, 10))
# annot=False because 43x43 is too dense for individual numbers
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix on Normal (Unaffected) Test Data', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.show()

print("\n--- Script Finished ---")

HOG LOGISTIC REGRESSION ON NORMAL DATA
:::::::::::::::::::::::::::::::::
:::::::::::::::::::::::::::::::::

In [ ]:
from skimage.feature import hog
from skimage import color
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import cv2
import os

# Load and preprocess dataset
X = []
y = []

# Replace with your actual dataset loading logic
for img_path, label in your_dataset:
    img = cv2.imread(img_path)
    img = cv2.resize(img, (64, 64))
    gray = color.rgb2gray(img)

    # Extract HOG features
    features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys')
    
    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"HOG + Logistic Regression Accuracy: {acc:.4f}")


HOG LR TEST
:::::::::::::::;
:::::::::::::::::

In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2
from skimage.feature import hog  # assuming skimage is installed
from skimage import exposure
# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]



# --- Predict on External Images using HOG ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64, 64))

        # Extract HOG features (same as used during training)
        hog_features = hog(img, orientations=9, pixels_per_cell=(8, 8),
                           cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

        predicted_label = model.predict([hog_features])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")

print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


SVM ON NORMAL DATA
:::::::::::::::::::
:::::::::::::::::::

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt

X = []  
y = []  

# Replace this with your actual dataset loading
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = SVC(kernel='rbf', C=10, gamma='scale')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Macro and Weighted Scores
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")


SVM NORMAL TEST
:::::::::::::::
:::::::::::::::

In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]

# --- Predict on External Images ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")
print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


SVM MODEL OLD
::::::::::::::
::::::::::::::::

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt


X = []  
y = []  

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = SVC(kernel='poly', degree=3,C=10, gamma='scale')  
model.fit(X_train, y_train)


y_pred = model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))


print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))


cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


KNN NORMAL
::::::::::
;::::::::::

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import cv2
import os


X = []  # features
y = []  # labels

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = KNeighborsClassifier(n_neighbors=5)  # k=5
model.fit(X_train, y_train)


y_pred = model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))


print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()



ACTUAL KNN
::::::::::::
::::::::::::::
;:::::::::::::;

In [ ]:
#modified KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt

X = []  # features
y = []  # labels

# Replace `your_dataset` with actual list of (img_path, label)
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Macro and Weighted Scores
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Macro and Weighted Scores
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")


KNN TEST
::::::::::

In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]

# --- Predict on External Images ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")
print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


RF1
:::::
;;::::

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import cv2
import os


X = []  # features
y = []  # labels

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (32, 32))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))


print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()



RF HOG
::::::
::::::::

In [ ]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- NEW: Import the HOG function ---
from skimage.feature import hog

# Assume 'your_dataset' is a list of tuples like [(img_path, label), ...]
# This part is just for making the script runnable as a placeholder.
# Replace this with your actual data loading logic.
print("NOTE: Using placeholder data. Replace with your 'your_dataset' loading.")
# --- Start Placeholder ---
if not os.path.exists('GTSRB/Train'):
    print("Error: GTSRB dataset not found. This script will not run without it.")
    exit()
your_dataset = []
for label_folder in range(43):
    folder_path = f'GTSRB/Train/{label_folder}'
    if os.path.exists(folder_path):
        for img_file in os.listdir(folder_path)[:50]: # Taking a subset for speed
            if img_file.endswith('.ppm'):
                your_dataset.append((os.path.join(folder_path, img_file), label_folder))
# --- End Placeholder ---


X_features = []  # To store HOG features
y_labels = []    # To store labels

print("Extracting HOG features from images...")
for img_path, label in your_dataset:
    # 1. Load image in grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    
    # 2. Resize image (HOG parameters depend on image size)
    img = cv2.resize(img, (32, 32))

    # --- KEY CHANGE: Calculate HOG features instead of flattening pixels ---
    # The parameters (orientations, pixels_per_cell, etc.) can be tuned
    # for better performance. These are common starting values.
    features = hog(img, 
                   orientations=9, 
                   pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), 
                   block_norm='L2-Hys', 
                   feature_vector=True)
    
    X_features.append(features)
    y_labels.append(label)

print(f"Feature extraction complete. Feature vector length: {len(X_features[0])}")

X = np.array(X_features)
y = np.array(y_labels)

# The rest of the pipeline is identical
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training RandomForestClassifier on {len(X_train)} samples...")
# Using more estimators is common for HOG features
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Evaluating model...")
y_pred = model.predict(X_test)

# --- Evaluation ---
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
# Generate target names for the report
class_names = [str(i) for i in range(len(np.unique(y)))] 
print(classification_report(y_test, y_pred, target_names=class_names))

# --- Visualization ---
print("Generating Confusion Matrix...")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues") # annot=False for dense matrices
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Random Forest with HOG Features')
plt.show()

In [ ]:
KNN LR RF 
:::::::::
::::::::::

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt

X = []
y = []

# Replace with your actual dataset loading
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Base learners
base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True))  # `probability=True` needed for stacking
]

# Meta-model
meta_model = LogisticRegression(max_iter=1000)

# Stacking model
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_model.fit(X_train, y_train)

# Prediction
y_pred = stacking_model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")


In [ ]:
TEST
:::::
:::::

In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]

# --- Predict on External Images ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")
print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
LR RF

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, precision_score,
    recall_score, f1_score, confusion_matrix
)
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt

X = []
y = []

# Replace with your actual dataset
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Only Random Forest as base model
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
]

# Logistic Regression as meta-model
meta_model = LogisticRegression(max_iter=1000)

# Stacking Classifier
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_model.fit(X_train, y_train)

# Predict
y_pred = stacking_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Precision, Recall, F1
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")
add toatl( precision and recall f1 score) and 

LR RF HOG STACKING 
;;;;;;;;;;;;;;
;;;;;;;;;;;

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, precision_score,
    recall_score, f1_score, confusion_matrix
)
from skimage.feature import hog
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt

X = []
y = []

# Replace with your actual dataset
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    
    # HOG feature extraction
    hog_features = hog_features = hog(img, orientations=12, pixels_per_cell=(4, 4),
                   cells_per_block=(2, 2), block_norm='L2-Hys')
                    #    """hog(img, orientations=9, pixels_per_cell=(8, 8),
                    #    cells_per_block=(2, 2), block_norm='L2-Hys')"""
    
    X.append(hog_features)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Only Random Forest as base model
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
]

# Logistic Regression as meta-model
meta_model = LogisticRegression(max_iter=1000)

# Stacking Classifier
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_model.fit(X_train, y_train)

# Predict
y_pred = stacking_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Precision, Recall, F1
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")

# Total scores across all classes
total_precision_macro = precision_macro * 43
total_recall_macro = recall_macro * 43
total_f1_macro = f1_macro * 43

total_precision_weighted = precision_weighted * 43
total_recall_weighted = recall_weighted * 43
total_f1_weighted = f1_weighted * 43

print(f"\nTotal Macro Precision: {total_precision_macro:.2f}")
print(f"Total Macro Recall: {total_recall_macro:.2f}")
print(f"Total Macro F1 Score: {total_f1_macro:.2f}")

print(f"\nTotal Weighted Precision: {total_precision_weighted:.2f}")
print(f"Total Weighted Recall: {total_recall_weighted:.2f}")
print(f"Total Weighted F1 Score: {total_f1_weighted:.2f}")


!!!!hog CNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.feature import hog
from skimage.color import rgb2gray
from PIL import Image
import numpy as np

# ---------- 1️⃣ Dataset: CNN + HOG -------------
class GTSRBHOGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img_np = np.array(img)
        gray = rgb2gray(img_np)

        hog_feature = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        hog_feature = torch.tensor(hog_feature, dtype=torch.float32)

        if self.transform:
            img_tensor = self.transform(img)
        else:
            img_tensor = transforms.ToTensor()(img)

        label = self.labels[idx]
        return img_tensor, hog_feature, label

# ---------- 2️⃣ Model: CNN + HOG Fusion -------------
class CNN_HOG_Model(nn.Module):
    def __init__(self, hog_dim, num_classes=43):
        super(CNN_HOG_Model, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Assuming input image 32x32 -> pool -> 16x16 -> pool -> 8x8
        self.cnn_out_dim = 64 * 8 * 8

        self.fc1 = nn.Linear(self.cnn_out_dim + hog_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x_img, x_hog):
        x = F.relu(self.conv1(x_img))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.cat((x, x_hog), dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------- 3️⃣ Dummy paths & labels (replace!) -------------
# This is placeholder: Replace with your real file paths and labels
# Load sample image paths from GTSRB/Train if available
import glob
gtsrb_imgs = glob.glob('GTSRB/Train/*/*.png') + glob.glob('GTSRB/Train/*/*.ppm')
if len(gtsrb_imgs) >= 2:
    train_image_paths = gtsrb_imgs[:10]
    train_labels = [0] * len(train_image_paths)
else:
    train_image_paths = []
    train_labels = []


# ---------- 4️⃣ Transforms -------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# ---------- 5️⃣ Loaders -------------
train_dataset = GTSRBHOGDataset(train_image_paths, train_labels, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# ---------- 6️⃣ Initialize -------------
# Get HOG dim from one sample
hog_dim = len(hog(np.zeros((32, 32)), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)))
model = CNN_HOG_Model(hog_dim=hog_dim, num_classes=43)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ---------- 7️⃣ Training Loop -------------
for epoch in range(2):  # example epochs
    model.train()
    for images, hogs, labels in train_loader:
        images = images.to(device)
        hogs = hogs.to(device)
        labels = labels.to(device)

        outputs = model(images, hogs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}], Loss: {loss.item():.4f}")

print("✅ Done")


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.feature import hog
from skimage.color import rgb2gray
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split

# ---------- ✅ 1️⃣ EXTRACT IMAGE PATHS + LABELS ----------
your_dataset = []
base_folder = "GTSRB/Train"  # update with your actual path

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))

# Split into separate lists
image_paths = [x[0] for x in your_dataset]
labels = [x[1] for x in your_dataset]

# ---------- ✅ 2️⃣ TRAIN / VAL SPLIT ----------
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

# ---------- ✅ 3️⃣ DATASET: HOG + IMAGE ----------
class GTSRBHOGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img = img.resize((32, 32))
        img_np = np.array(img)
        gray = rgb2gray(img_np)

        hog_feature = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        hog_feature = torch.tensor(hog_feature, dtype=torch.float32)

        if self.transform:
            img_tensor = self.transform(img)
        else:
            img_tensor = transforms.ToTensor()(img)

        label = self.labels[idx]
        return img_tensor, hog_feature, label

# ---------- ✅ 4️⃣ TRANSFORM ----------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# ---------- ✅ 5️⃣ DATALOADERS ----------
train_dataset = GTSRBHOGDataset(train_paths, train_labels, transform=transform)
val_dataset = GTSRBHOGDataset(val_paths, val_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# ---------- ✅ 6️⃣ MODEL: CNN + HOG Fusion ----------
class CNN_HOG_Model(nn.Module):
    def __init__(self, hog_dim, num_classes=43):
        super(CNN_HOG_Model, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.cnn_out_dim = 64 * 8 * 8  # for 32x32 input

        self.fc1 = nn.Linear(self.cnn_out_dim + hog_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x_img, x_hog):
        x = F.relu(self.conv1(x_img))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.cat((x, x_hog), dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------- ✅ 7️⃣ INIT ----------
hog_dim = len(hog(np.zeros((32, 32)), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)))
model = CNN_HOG_Model(hog_dim=hog_dim, num_classes=43)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ---------- ✅ 8️⃣ TRAIN LOOP ----------
for epoch in range(2):  # change to more epochs!
    model.train()
    for images, hogs, labels in train_loader:
        images = images.to(device)
        hogs = hogs.to(device)
        labels = labels.to(device)

        outputs = model(images, hogs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}], Loss: {loss.item():.4f}")

print("✅ DONE with your GTSRB + CNN + HOG pipeline")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ---------- ✅ 1️⃣ Validation Loop -------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, hogs, labels in val_loader:
        images = images.to(device)
        hogs = hogs.to(device)
        labels = labels.to(device)

        outputs = model(images, hogs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- ✅ 2️⃣ Compute Metrics -------------
accuracy = accuracy_score(all_labels, all_preds)
precision_macro = precision_score(all_labels, all_preds, average='macro')
recall_macro = recall_score(all_labels, all_preds, average='macro')
f1_macro = f1_score(all_labels, all_preds, average='macro')

precision_weighted = precision_score(all_labels, all_preds, average='weighted')
recall_weighted = recall_score(all_labels, all_preds, average='weighted')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f"\nValidation Accuracy: {accuracy:.4f}")

print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"Weighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds))


!!!cnn hog without problematic class!!

In [ ]:
import os
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models

from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ---------------------------
# 1️⃣ CONFIG
# ---------------------------
DATA_DIR = 'GTSRB/Train' if os.path.exists('GTSRB/Train') else 'GTSRB/Train'  # your data folder
REMOVE_CLASSES = ['0', '19', '32', '37', '41']

# ---------------------------
# 2️⃣ CNN PART
# ---------------------------

# Transform for CNN
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Load all data first
full_dataset = datasets.ImageFolder(root="GTSRB/Train", transform=transform)

# Figure out which indices to remove
class_to_idx = full_dataset.class_to_idx
remove_indices = [class_to_idx[c] for c in REMOVE_CLASSES if c in class_to_idx]

# Keep only samples not in remove list
keep_indices = [i for i, (_, label) in enumerate(full_dataset.samples) if label not in remove_indices]

filtered_dataset = Subset(full_dataset, keep_indices)
train_loader = DataLoader(filtered_dataset, batch_size=32, shuffle=True)

# Simple CNN using ResNet18
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, len(full_dataset.classes) - len(remove_indices))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"CNN Training with {len(keep_indices)} images and {len(full_dataset.classes) - len(remove_indices)} classes")

# One quick training loop example
model.train()
for epoch in range(1):  # keep small for test
    total_loss = 0.0
    for images, labels in train_loader:
        outputs = model(images)
        # Remap labels to new indices
        new_labels = torch.tensor([list(sorted(set(range(len(full_dataset.classes))) - set(remove_indices))).index(l.item()) for l in labels])
        loss = criterion(outputs, new_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}] Loss: {total_loss:.4f}")

# ---------------------------
# 3️⃣ HOG + SVM PART
# ---------------------------

print("\nHOG + SVM training...")

hog = cv2.HOGDescriptor()
X, y = [], []

# Get valid class folders
class_folders = [f for f in os.listdir(DATA_DIR) if f not in REMOVE_CLASSES and os.path.isdir(os.path.join(DATA_DIR, f))]

print(f"Using classes: {class_folders}")

# Load images + compute HOG features
for class_idx, class_name in enumerate(sorted(class_folders)):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (64, 128))
        features = hog.compute(img).flatten()
        X.append(features)
        y.append(class_idx)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = svm.SVC()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"HOG + SVM Accuracy: {acc:.4f}")


!!!hog svm

In [ ]:
import os
import cv2
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, precision_score,
    recall_score, f1_score, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG
# ---------------------------
DATA_DIR = 'GTSRB/Train'  # your dataset folder

# ---------------------------
# INIT
# ---------------------------
hog = cv2.HOGDescriptor()
X, y = [], []

# ---------------------------
# LOAD DATA
# ---------------------------
class_folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]
class_folders = sorted(class_folders)
print("Classes used:", class_folders)

for class_idx, class_name in enumerate(class_folders):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (64, 128))
        features = hog.compute(img).flatten()
        X.append(features)
        y.append(class_idx)

X = np.array(X)
y = np.array(y)
print(f"Total samples: {len(X)}")

# ---------------------------
# SPLIT DATA
# ---------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

# ---------------------------
# TRAIN SVM
# ---------------------------
clf = svm.SVC(kernel='rbf')
clf.fit(X_train, y_train)

# ---------------------------
# PREDICT
# ---------------------------
y_train_pred = clf.predict(X_train)
y_val_pred = clf.predict(X_val)

# ---------------------------
# TRAINING ACCURACY
# ---------------------------
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_acc:.4f}")

# ---------------------------
# VALIDATION ACCURACY
# ---------------------------
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

# ---------------------------
# DETAILED METRICS
# ---------------------------
precision_macro = precision_score(y_val, y_val_pred, average='macro')
recall_macro = recall_score(y_val, y_val_pred, average='macro')
f1_macro = f1_score(y_val, y_val_pred, average='macro')

precision_weighted = precision_score(y_val, y_val_pred, average='weighted')
recall_weighted = recall_score(y_val, y_val_pred, average='weighted')
f1_weighted = f1_score(y_val, y_val_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")

# ---------------------------
# CLASSIFICATION REPORT
# ---------------------------
print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=class_folders))

# ---------------------------
# CONFUSION MATRIX
# ---------------------------
cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_folders,
            yticklabels=class_folders)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()
